Data Loading: pandas.read_csv
Data Exploration: DataFrame.shape, DataFrame.columns, DataFrame.dtypes, DataFrame.isnull().sum(), DataFrame.duplicated().sum()
Feature and Target Separation
Data Splitting
Data Standardization
Data Normalization
Filter Method Feature Selection
Wrapper Method Feature Selection

 !nvdia-smi
We are going to perfom some of this EDA techniques on the 'Titanic" dataset 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns 
import scipy as sp


In [2]:


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.linear_model import LogisticRegression


# Load the Titanic dataset
df = sns.load_dataset('titanic')

print(df.head())


# Check the dataset
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


# Prepare the data
data = df.copy()

# Remove columns that are not being used
data = data.drop(
    columns=['alive', 'class', 'who', 'adult_male', 'deck', 'embark_town'],
    errors='ignore'
)

# Fill missing values
data['age'] = data['age'].fillna(data['age'].median())
data['embarked'] = data['embarked'].fillna(data['embarked'].mode()[0])

# Convert categorical columns into numerical columns
data = pd.get_dummies(
    data,
    columns=['sex', 'embarked'],
    drop_first=True
)

# Convert boolean columns into integers
data = data.astype(int)

print("\nProcessed data:")
print(data.head())


# Separate features and target
X = data.drop('survived', axis=1)
y = data['survived']

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print("survived")


# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# Standardization
standard_scaler = StandardScaler()

X_train_standardized = standard_scaler.fit_transform(X_train)
X_test_standardized = standard_scaler.transform(X_test)

X_train_standardized = pd.DataFrame(
    X_train_standardized,
    columns=X_train.columns
)

X_test_standardized = pd.DataFrame(
    X_test_standardized,
    columns=X_test.columns
)

print("\nStandardized data:")
print(X_train_standardized.head())


# Normalization
normal_scaler = MinMaxScaler()

X_train_normalized = normal_scaler.fit_transform(X_train)
X_test_normalized = normal_scaler.transform(X_test)

X_train_normalized = pd.DataFrame(
    X_train_normalized,
    columns=X_train.columns
)

X_test_normalized = pd.DataFrame(
    X_test_normalized,
    columns=X_test.columns
)

print("\nNormalized data:")
print(X_train_normalized.head())


# Filter method
filter_selector = SelectKBest(
    score_func=chi2,
    k=5
)

filter_selector.fit(X_train, y_train)

filter_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': filter_selector.scores_
})

filter_scores = filter_scores.sort_values(
    by='Score',
    ascending=False
)

selected_filter_features = X_train.columns[
    filter_selector.get_support()
]

print("\nFilter method results:")
print(filter_scores)

print("\nSelected features:")
print(selected_filter_features.tolist())


# Wrapper method
model = LogisticRegression(max_iter=1000)

rfe = RFE(
    estimator=model,
    n_features_to_select=5
)

rfe.fit(X_train, y_train)

rfe_results = pd.DataFrame({
    'Feature': X_train.columns,
    'Selected': rfe.support_,
    'Ranking': rfe.ranking_
})

print("\nWrapper method results:")
print(rfe_results)

print("\nSelected features:")
print(X_train.columns[rfe.support_].tolist())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
Shape: (891, 15)

Columns:
Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')

Da